In [1]:
import os
import json
import glob
from pathlib import Path
import pyarrow.parquet as pq
from tokenizers.models import BPE
from datasets import load_from_disk
from tokenizers.trainers import BpeTrainer
from tokenizers import Tokenizer, AddedToken
from tokenizers.pre_tokenizers import ByteLevel
from tokenizers.decoders import ByteLevel as ByteLevelDecoder
from huggingface_hub import snapshot_download

In [2]:
VOCAB_SIZE = 32_000
MIN_FREQUENCY = 2
BATCH_SIZE = 1_000

OUTPUT_DIR = Path("tokenizer")

In [ ]:
snapshot_download('HuggingFaceFW/fineweb-edu', repo_type='dataset',
                  allow_patterns='sample/10BT/00[0-6]_00000.parquet')

In [3]:
FILES = sorted(glob.glob(os.path.expanduser(
    "~/.cache/huggingface/hub/datasets--HuggingFaceFW--fineweb-edu"
    "/snapshots/*/sample/10BT/*.parquet")))

In [4]:
def iter_documents_parquet(files, batch_size=1000):
    for f in files:
        for batch in pq.ParquetFile(f).iter_batches(batch_size=batch_size, columns=["text"]):
            for t in batch.column("text").to_pylist():
                t = t.strip()
                if t:
                    yield t

In [5]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [6]:
SPECIAL_TOKENS = [
    "<|beginoftext|>",
    "<|endoftext|>",
    "<|unk|>",
]

In [7]:
tokenizer = Tokenizer(BPE(unk_token="<|unk|>"))

In [8]:
tokenizer.pre_tokenizer = ByteLevel(add_prefix_space=False)

In [9]:
tokenizer.pre_tokenizer = ByteLevel(add_prefix_space=False)

In [10]:
tokenizer.decoder = ByteLevelDecoder()

In [11]:
trainer = BpeTrainer(
        vocab_size=VOCAB_SIZE,
        min_frequency=MIN_FREQUENCY,
        special_tokens=[
            AddedToken(token, special=True)
            for token in SPECIAL_TOKENS
        ],
        initial_alphabet=ByteLevel.alphabet(),
        show_progress=True,
    )

In [12]:
%%time
tokenizer.train_from_iterator(
        iter_documents_parquet(FILES),
        trainer=trainer,
    )




CPU times: user 4h 11min 53s, sys: 1min 50s, total: 4h 13min 43s
Wall time: 12min


In [13]:
tokenizer_path = OUTPUT_DIR / "tokenizer.json"
tokenizer.save(str(tokenizer_path))

In [14]:
token_ids = {
        token: tokenizer.token_to_id(token)
        for token in SPECIAL_TOKENS
    }

with (OUTPUT_DIR / "special_tokens.json").open("w", encoding="utf-8") as f:
    json.dump(token_ids, f, indent=2)

In [15]:
print("Vocabulary size:", tokenizer.get_vocab_size())
print("Tokenizer saved to:", tokenizer_path)
print("Special-token IDs:", token_ids)

Vocabulary size: 32000
Tokenizer saved to: tokenizer/tokenizer.json
Special-token IDs: {'<|beginoftext|>': 0, '<|endoftext|>': 1, '<|unk|>': 2}


In [16]:
tokenizer = Tokenizer.from_file(str(tokenizer_path))

In [20]:
tests = [
    "Once upon a time, there was a little fox.",
    "DATA URL: https://huggingface.co/datasets/HuggingFaceFW/fineweb-edu",
    "def f(x):\n    return [i**2 for i in range(x) if i % 3 == 0]",
    "In 2023, revenue grew 12.7% to $4,821,000.55 (up from 3.1e6).",
]


In [21]:
for text in tests:
    encoded = tokenizer.encode(text)
    decoded = tokenizer.decode(encoded.ids)

    assert decoded == text, (
        f"Round-trip failed: {text!r} -> {decoded!r}"
    )

    print("Text:", text)
    print("Tokens:", encoded.tokens)
    print("IDs:", encoded.ids)
    print("Decoded:", decoded)
    print("-" * 40)

Text: Once upon a time, there was a little fox.
Tokens: ['Once', 'Ġupon', 'Ġa', 'Ġtime', ',', 'Ġthere', 'Ġwas', 'Ġa', 'Ġlittle', 'Ġfox', '.']
IDs: [7035, 2188, 260, 653, 14, 640, 416, 260, 1670, 15404, 16]
Decoded: Once upon a time, there was a little fox.
----------------------------------------
Text: DATA URL: https://huggingface.co/datasets/HuggingFaceFW/fineweb-edu
Tokens: ['D', 'ATA', 'ĠURL', ':', 'Ġhttps', '://', 'h', 'ugging', 'face', '.', 'co', '/', 'd', 'atas', 'ets', '/', 'H', 'ugging', 'F', 'ace', 'F', 'W', '/', 'f', 'ine', 'web', '-', 'edu']
IDs: [38, 26943, 14252, 28, 5113, 2102, 74, 20284, 1754, 16, 3525, 17, 70, 29166, 1333, 17, 42, 20284, 40, 583, 40, 57, 17, 72, 469, 18241, 15, 6130]
Decoded: DATA URL: https://huggingface.co/datasets/HuggingFaceFW/fineweb-edu
----------------------------------------
Text: def f(x):
    return [i**2 for i in range(x) if i % 3 == 0]
Tokens: ['def', 'Ġf', '(', 'x', '):', 'Ċ', 'Ġ', 'Ġ', 'Ġ', 'Ġreturn', 'Ġ[', 'i', '**', '2', 'Ġfor', 'Ġi', '

In [19]:
for token in [
    "<|beginoftext|>",
    "<|endoftext|>",
    "<|unk|>",
]:
    assert tokenizer.token_to_id(token) is not None